# InstaNexus demo — assembling a nanobody from de novo PSMs

This notebook walks through the **InstaNexus** workflow on a real nanobody
dataset (`nb1`): starting from raw peptide-spectrum match (PSM) predictions
produced by a de novo sequencing tool (e.g. InstaNovo), it preprocesses the
data, assembles overlapping peptides into scaffolds with a De Bruijn graph
assembler, clusters and aligns the scaffolds, and derives consensus sequences
that approximate the original protein chain.

By the end of this notebook you will have:

1. Checked that all required tools are available.
2. Run the `instanexus` CLI end-to-end on `inputs/nb1.csv`.
3. Inspected the assembled scaffolds, consensus sequences, peptide coverage,
   and sequence logos produced along the way.

Feel free to change the parameters in section 2 and re-run the pipeline to see
how they affect the final assembly.

## 1. Environment check

InstaNexus relies on a handful of external tools in addition to the Python
package itself:

- **Python** (≥ 3.10) and the `instanexus` CLI (installed via `uv sync`)
- **MMseqs2** for clustering scaffolds (`clustering.py`)
- **Clustal Omega** (`clustalo`) for multiple sequence alignment (`alignment.py`)

The cell below checks that each of these is reachable on the `PATH`. If you
are running inside the provided devcontainer or Binder image, everything
should already be installed.

In [ ]:
import shutil
import subprocess
import sys

tools = {
    "python": [sys.executable, "--version"],
    "mmseqs": ["mmseqs", "version"],
    "clustalo": ["clustalo", "--version"],
    "instanexus": ["instanexus", "--help"],
}

for name, cmd in tools.items():
    path = shutil.which(cmd[0])
    if path is None:
        print(f"[MISSING] {name}: '{cmd[0]}' not found on PATH")
        continue
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, check=False)
        first_line = (result.stdout or result.stderr).strip().splitlines()[0]
        print(f"[OK] {name}: {first_line}  ({path})")
    except Exception as exc:
        print(f"[ERROR] {name}: {exc}")


## 2. Parameters

All the paths and pipeline parameters used in this notebook are collected here
so they are easy to find and modify. Paths are resolved relative to the
repository root (`BASE_DIR`).

In [ ]:
from pathlib import Path

# Resolve the repository root whether we run as a script or inside a notebook
try:
    BASE_DIR = Path(__file__).resolve().parents[1]
except NameError:
    BASE_DIR = Path().resolve()
    while BASE_DIR.name != "InstaNexus" and BASE_DIR != BASE_DIR.parent:
        BASE_DIR = BASE_DIR.parent

# --- Paths -------------------------------------------------------------
INPUT_CSV = BASE_DIR / "inputs" / "nb1.csv"                 # raw PSM predictions (de novo output)
METADATA_JSON = BASE_DIR / "json" / "sample_metadata.json"  # protein sequence, chain(s), proteases for nb1
CONTAM_FASTA = BASE_DIR / "fasta" / "contaminants.fasta"    # contaminant sequences removed before assembly
OUTPUT_DIR = BASE_DIR / "outputs" / "demo_nb1"              # where the pipeline writes its results

# --- Pipeline parameters ------------------------------------------------
assembly_mode = "dbg"     # assembly algorithm: "dbg" builds a De Bruijn graph from overlapping k-mers
conf = 0.9                # minimum confidence score: PSMs below this are discarded during preprocessing
kmer_size = 7             # length of the k-mers used to build the De Bruijn graph (only used by dbg* modes)
min_overlap = 3           # minimum overlap (in residues) required to connect two reads/contigs
size_threshold = 12       # minimum contig length kept after assembly (shorter contigs are dropped)
min_seq_id = 0.85         # minimum sequence identity (MMseqs2 -min-seq-id) used when clustering scaffolds
coverage = 0.8            # minimum alignment coverage (MMseqs2 -c) used when clustering scaffolds

print(f"BASE_DIR      = {BASE_DIR}")
print(f"INPUT_CSV     = {INPUT_CSV}")
print(f"METADATA_JSON = {METADATA_JSON}")
print(f"CONTAM_FASTA  = {CONTAM_FASTA}")
print(f"OUTPUT_DIR    = {OUTPUT_DIR}")


## 3. Inspecting the input

`inputs/nb1.csv` contains raw PSM predictions: one row per spectrum, with the
predicted peptide sequence (`prediction_untokenised`) and a calibrated
confidence score (`calibrated_confidence`). Let's load it and look at the
distribution of confidence scores — this is what the `conf` threshold (set
above) filters on during preprocessing.

In [ ]:
import pandas as pd

df_raw = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df_raw)} PSMs from {INPUT_CSV.name}")
df_raw[["spectrum_id", "prediction_untokenised", "calibrated_confidence"]].head()


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(df_raw["calibrated_confidence"], bins=40, color="#4C72B0", edgecolor="white")
ax.axvline(conf, color="crimson", linestyle="--", label=f"conf threshold = {conf}")
ax.set_xlabel("Calibrated confidence score")
ax.set_ylabel("Number of PSMs")
ax.set_title("Distribution of PSM confidence scores (nb1)")
ax.legend()
fig.tight_layout()
plt.show()


## 4. Running the InstaNexus pipeline

We now call the `instanexus` CLI as a subprocess with the parameters defined
above. This runs the full pipeline — preprocessing, assembly, clustering,
alignment, and consensus generation — and writes everything to `OUTPUT_DIR`.

`--metadata-json-path` and `--contaminants-fasta-path` are optional flags: they
enable protease-aware splitting, chain filtering, and contaminant removal. They
are provided here because `sample_metadata.json` and `contaminants.fasta` are
available for the `nb1` sample, but the pipeline also runs without them in
"minimal mode" for de novo outputs that don't carry InstaNovo-specific columns.

In [ ]:
import subprocess

cmd = [
    "instanexus",
    "--input-csv", str(INPUT_CSV),
    "--metadata-json-path", str(METADATA_JSON),
    "--contaminants-fasta-path", str(CONTAM_FASTA),
    "--output-dir", str(OUTPUT_DIR),
    "--assembly-mode", assembly_mode,
    "--conf", str(conf),
    "--kmer-size", str(kmer_size),
    "--min-overlap", str(min_overlap),
    "--size-threshold", str(size_threshold),
    "--min-seq-id", str(min_seq_id),
    "--coverage", str(coverage),
]

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout[-4000:])
if result.returncode != 0:
    print(result.stderr[-4000:])
print(f"\nExit code: {result.returncode}")


## 5. Inspecting the outputs

The pipeline writes a predictable folder structure under `OUTPUT_DIR`:

```
OUTPUT_DIR/
├── cleaned.csv                       # preprocessed PSMs
├── summary.tsv                       # one-line run summary
└── scaffolds/
    ├── scaffolds.fasta               # assembled scaffolds
    ├── clustering/                   # MMseqs2 clusters
    ├── alignment/                    # Clustal Omega alignments (.afa)
    └── consensus/
        ├── consensus_fasta/          # consensus sequence per scaffold
        ├── consensus_stats.json
        ├── heatmap/                  # per-scaffold coverage heatmaps
        └── logo/                     # per-scaffold sequence logos
```

Let's list what was generated and load the consensus sequences with
BioPython.

In [ ]:
scaffolds_dir = OUTPUT_DIR / "scaffolds"
consensus_dir = scaffolds_dir / "consensus"
consensus_fasta_dir = consensus_dir / "consensus_fasta"

print(f"Generated files under {OUTPUT_DIR.relative_to(BASE_DIR)}:")
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(" ", path.relative_to(OUTPUT_DIR))


In [ ]:
from Bio import SeqIO

consensus_records = []
for fasta_path in sorted(consensus_fasta_dir.glob("*_consensus.fasta")):
    consensus_records.extend(SeqIO.parse(fasta_path, "fasta"))

print(f"Found {len(consensus_records)} consensus sequence(s)\n")
for record in consensus_records:
    print(f">{record.id}  (length={len(record.seq)})")
    print(record.seq)
    print()


## 6. Coverage plot

The `heatmap/` folder contains, for each scaffold, a positional view of how
many peptides cover each residue of the consensus sequence — this is a quick
way to spot well-supported regions versus low-coverage gaps. We display the
heatmap for the first scaffold below.

In [ ]:
import matplotlib.image as mpimg

heatmap_dir = consensus_dir / "heatmap"
heatmap_files = sorted(heatmap_dir.glob("*_heatmap.svg"))

if heatmap_files:
    print(f"Showing coverage heatmap: {heatmap_files[0].name}")
    # SVGs are rendered natively by Jupyter when displayed via IPython
    from IPython.display import SVG, display

    display(SVG(filename=str(heatmap_files[0])))
else:
    print("No heatmap files found — re-run the pipeline without --skip-plots.")


## 7. Sequence logo

`logomaker` is used to draw a sequence logo from the aligned peptides
(`alignment/*_aligned.afa`) supporting each scaffold, highlighting the
per-position amino-acid composition and conservation. Below we regenerate the
logo for the first aligned scaffold directly from its alignment FASTA.

In [ ]:
import logomaker
from Bio import AlignIO

alignment_dir = scaffolds_dir / "alignment"
afa_files = sorted(alignment_dir.glob("*_aligned.afa"))

if afa_files:
    alignment = AlignIO.read(afa_files[0], "fasta")
    counts_df = logomaker.alignment_to_matrix(sequences=[str(rec.seq) for rec in alignment])

    fig, ax = plt.subplots(figsize=(10, 3))
    logomaker.Logo(counts_df, ax=ax, color_scheme="chemistry")
    ax.set_title(f"Sequence logo — {afa_files[0].stem}")
    ax.set_xlabel("Alignment position")
    ax.set_ylabel("Amino-acid count")
    fig.tight_layout()
    plt.show()
else:
    print("No alignment files found under", alignment_dir)


## 8. Summary

A quick textual recap of this run, read back from `summary.tsv` and
`consensus_stats.json`.

In [ ]:
import json

summary_path = OUTPUT_DIR / "summary.tsv"
stats_path = consensus_dir / "consensus_stats.json"

print(f"Run: {INPUT_CSV.stem}  |  assembly_mode={assembly_mode}  |  conf={conf}  |  kmer_size={kmer_size}")
print(f"Results written to: {OUTPUT_DIR}\n")

if summary_path.exists():
    summary_df = pd.read_csv(summary_path, sep="\t")
    print("--- summary.tsv ---")
    print(summary_df.T)

if stats_path.exists():
    with open(stats_path) as f:
        stats = json.load(f)
    print("\n--- consensus_stats.json ---")
    print(json.dumps(stats, indent=2))

print(f"\nAssembled {len(consensus_records)} consensus sequence(s) from {len(df_raw)} raw PSMs "
      f"(confidence threshold = {conf}).")
print("Try changing assembly_mode, kmer_size, min_overlap, or size_threshold in section 2 "
      "and re-running the pipeline to see how the assembled scaffolds change.")
